# Enviroweather API documentation (v1)

## Step 4a: More Complex Result Model Parameters

Covers `applescab`, `orientalfruitmoth`, and `tomcast`. Unlike the models in Step 4, these don't have constant defaults - the `defaultValue` for their inputs is `null`, but the group has a `defaultResourceKey`.

To get the real default, you make a *second* RM-API request using `defaultResourceKey` as the `resultModelCode`, for the station/date the user picked. That response gives you a date (e.g. `gtStart_estimated`) to pre-fill the input with. The user can still override it.

In [ ]:
# this is a Marimo document, not a jupyter notebook.  Import marimo to use it
import marimo as mo

In [ ]:
import requests

from ewx_client import DEFAULT_STATION_CODE, ENVIRONMENTS, get_auth_header, get_environment_urls, get_site_token

## Setup: choose an environment and get a token

See [`ewx_api_v1_step1_tokens.py`](./ewx_api_v1_step1_tokens.py) for a full walkthrough of tokens; here we just reuse the `ewx_client` helpers.

In [ ]:
environment_names = list(ENVIRONMENTS.keys())
env_key_selector = mo.ui.dropdown(options=environment_names, label="choose environment")
env_key_selector

In [ ]:
if env_key_selector.selected_key:
    ewx_env = env_key_selector.selected_key
else:
    ewx_env = "production"

In [ ]:
api_url, rm_api_url = get_environment_urls(ewx_env)

In [ ]:
anonymous_token = get_site_token(api_url)
token_header = get_auth_header(anonymous_token)

In [ ]:
mo.md(rf"""
working with **{ewx_env}** environment for this session with base URLs

* **api**: {api_url}
* **rm api**: {rm_api_url}
""")

In [ ]:
from datetime import date, timedelta

# Weather-dependent RM-API calls use a couple of days ago, since "today's"
# observations may not be finalized yet.
recent_date = (date.today() - timedelta(days=2)).isoformat()

## Apple scab

`GET {api_url}/db2/resources/applescab/inputs?show=rmInputs` describes the `applescabbiofix` (aka "green tip") input - its `defaultResourceKey` is `mcintoshgreentip`.

To compute the actual default date for a given station/date, call RM-API with `resultModelCode=mcintoshgreentip`:

`GET {rm_api_url}/db2/run?stationCode=...&stationType=1&selectDate=...&resultModelCode=mcintoshgreentip`

The response's `gtStart_estimated` field is the date to pre-fill for `gtStart` in the real Step 5a `applescab` request.

In [ ]:
applescab_inputs_url = f"{api_url}/db2/resources/applescab/inputs?show=rmInputs"
applescab_inputs_response = requests.get(applescab_inputs_url, headers=token_header, timeout=30)
applescab_inputs_response.json()

In [ ]:
mcintoshgreentip_url = (
    f"{rm_api_url}/db2/run?stationCode={DEFAULT_STATION_CODE}&stationType=1"
    f"&selectDate={recent_date}&resultModelCode=mcintoshgreentip"
)
mcintoshgreentip_response = requests.get(mcintoshgreentip_url, headers=token_header, timeout=30)
mcintoshgreentip_data = mcintoshgreentip_response.json()
mcintoshgreentip_data

In [ ]:
if not mcintoshgreentip_data["error"]:
    estimate_md = mo.md(
        f"estimated `gtStart` for apple scab: **{mcintoshgreentip_data['data']['gtStart_estimated']}**"
    )
else:
    estimate_md = mo.md(f"error: {mcintoshgreentip_data['message']}")
estimate_md

## Oriental fruit moth

Same pattern. `orientalfruitmoth`'s two biofix inputs use `defaultResourceKey = orientalfruitmothadultemergence`, which returns `biofix1_estimated` and `biofix2_estimated`.

In [ ]:
orientalfruitmoth_inputs_url = f"{api_url}/db2/resources/orientalfruitmoth/inputs?show=rmInputs"
orientalfruitmoth_inputs_response = requests.get(orientalfruitmoth_inputs_url, headers=token_header, timeout=30)
orientalfruitmoth_inputs_response.json()

In [ ]:
orientalfruitmoth_defaults_url = (
    f"{rm_api_url}/db2/run?stationCode={DEFAULT_STATION_CODE}&stationType=1"
    f"&selectDate={recent_date}&resultModelCode=orientalfruitmothadultemergence"
)
orientalfruitmoth_defaults_response = requests.get(orientalfruitmoth_defaults_url, headers=token_header, timeout=30)
orientalfruitmoth_defaults_response.json()

## Tomcast

Same pattern again, with `defaultResourceKey = tomcastdefaults`.

In [ ]:
tomcast_inputs_url = f"{api_url}/db2/resources/tomcast/inputs?show=rmInputs"
tomcast_inputs_response = requests.get(tomcast_inputs_url, headers=token_header, timeout=30)
tomcast_inputs_response.json()

In [ ]:
tomcast_defaults_url = (
    f"{rm_api_url}/db2/run?stationCode={DEFAULT_STATION_CODE}&stationType=1"
    f"&selectDate={recent_date}&resultModelCode=tomcastdefaults"
)
tomcast_defaults_response = requests.get(tomcast_defaults_url, headers=token_header, timeout=30)
tomcast_defaults_response.json()